# Clonamos el repositorio con los modelos y herramientas

In [1]:
!git clone https://github.com/dannasalazar11/Msc_thesis.git

Cloning into 'Msc_thesis'...
remote: Enumerating objects: 488, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 488 (delta 38), reused 0 (delta 0), pack-reused 427 (from 1)
Receiving objects: 100% (488/488), 50.49 MiB | 40.29 MiB/s, done.
Resolving deltas: 100% (315/315), done.


In [2]:
import sys
sys.path.append('/kaggle/working/Msc_thesis')

from gmrrnet_adhd.utils import get_segmented_data
from tensorflow.keras.mixed_precision import set_global_policy
set_global_policy('mixed_float16')

import tensorflow as tf
import numpy as np
import random
import os

# Establecer semilla
seed = 42

# Semillas para módulos principales
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)

2025-09-28 21:21:45.144915: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759094505.510787      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759094505.622579      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
import numpy as np
import random
from collections import defaultdict
from copy import deepcopy

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    cohen_kappa_score,
    roc_auc_score
)


def train_L24O_cv(model_builder, X, y, sbjs, model_args, compile_args, folds, model_name=''):
    all_fold_metrics = []
    models = {}

    for fold, (train_subjects, test_subjects) in enumerate(folds):
        print("-" * 50)
        print(f"Fold {fold+1}/{len(folds)}. Test subjects: {test_subjects}")
        print("-" * 50)

        train_idx = [i for i, sbj in enumerate(sbjs) if sbj in train_subjects]
        test_idx = [i for i, sbj in enumerate(sbjs) if sbj in test_subjects]

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        sbjs_test = [sbjs[i] for i in test_idx]

        # --- Build and Compile Model for each fold ---
        tf.keras.backend.clear_session() #<-- Clear session to prevent any state leakage
        
        # Re-set seeds for each fold for perfect reproducibility of weight initialization
        np.random.seed(seed + fold)
        random.seed(seed + fold)
        tf.random.set_seed(seed + fold)

        model = model_builder(**model_args)
        # Use a deepcopy to prevent the optimizer state from carrying over
        compile_args_local = deepcopy(compile_args)
        if callable(compile_args_local["optimizer"]):
            compile_args_local["optimizer"] = compile_args_local["optimizer"]()  # <-- aquí se reinicia
        model.compile(**compile_args_local)

        
        # --- Callbacks ---
        # EarlyStopping with restore_best_weights is crucial
        early_stopping = EarlyStopping(
            monitor='val_loss', patience=25, min_delta=1e-4, restore_best_weights=True, verbose=1
        )
        # ReduceLROnPlateau helps to fine-tune when learning stalls
        reduce_lr = ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1
        )

        # --- Train the Model ---
        model.fit(
            X_train, y_train,
            epochs=150,  #<-- Increased epochs to give LR scheduler more time to work
            validation_data=(X_test, y_test),
            verbose=0, #<-- Verbose=2 gives one line per epoch, cleaner log
            batch_size=16,
            callbacks=[early_stopping, reduce_lr]
        )

        # --- Predictions and Evaluation ---
        y_pred_probs = model.predict(X_test)
        print(y_pred_probs.shape)
        y_pred = np.argmax(y_pred_probs, axis=1)
        y_true = np.argmax(y_test, axis=1)

        # Overall fold metrics
        fold_metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
            'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'kappa': cohen_kappa_score(y_true, y_pred),
            'auc': roc_auc_score(y_true, y_pred_probs[:, 1]) # Use probabilities for AUC
        }
        print(f"\nFold {fold+1} Metrics: {fold_metrics}")
        all_fold_metrics.append(fold_metrics)
        models[fold] = model

        # Accuracy por sujeto de test
        subject_correct = defaultdict(list)
        for yt, yp, sbj in zip(y_true, y_pred, sbjs_test):
            subject_correct[sbj].append(int(yt == yp))

        subject_accuracies = {
            sbj: np.mean(subject_correct[sbj]) for sbj in subject_correct
        }

        print("Average accuracy per test subject:")
        for sbj in test_subjects:
            acc_sbj = subject_accuracies.get(sbj, None)
            if acc_sbj is not None:
                print(f"  {sbj}: {acc_sbj:.4f}")
                
        
    # --- Final Comprehensive Report ---
    print("\n" + "="*50)
    print("Cross-Validation Final Results")
    print("="*50)
    
    # Calculate mean and std dev for each metric
    mean_metrics = {}
    for key in all_fold_metrics[0].keys():
        values = [f[key] for f in all_fold_metrics]
        mean_metrics[f'mean_{key}'] = np.mean(values)
        mean_metrics[f'std_{key}'] = np.std(values)

    print("Individual Fold Accuracies:")
    for i, f in enumerate(all_fold_metrics):
        print(f"  Fold {i+1}: {f['accuracy']:.4f}")
        
    print("\nAverage Performance across all folds:")
    for key, value in mean_metrics.items():
        print(f"  {key}: {value:.4f}")
        
    return all_fold_metrics

# Importar base de datos segmentada (Segmentos de 4 seg con translape del 50%, es decir, de 2 seg)

In [4]:
X, y, sbjs = get_segmented_data()
X.shape, y.shape, len(sbjs)

((8213, 19, 512), (8213, 2), 8213)

# Importamos el modelo y definimos los hiperparámetros

In [5]:
from tensorflow.keras.losses import CategoricalCrossentropy, MeanSquaredError
from tensorflow.keras.optimizers import Adam
from gmrrnet_adhd.models.cnn_lstm_eegnet import cnn_lstm_eegnet

model_name = 'CNN_LSTM_EEGNet'
model_args = {
    "n_channels": 19,      # EEG electrodes
    "n_times": 512,        # time points per trial
    "n_classes": 2,        # binary task
    "temporal_kernel": 25, # ≈200 ms at 128 Hz
    "pool_size": 20,
    "pool_stride": 10
}

compile_args = {
    'loss': CategoricalCrossentropy(),
    'optimizer': lambda: Adam(1e-2),  # función que retorna un nuevo optimizador
    'metrics': ['categorical_accuracy']
}


model = cnn_lstm_eegnet(**model_args)

model.summary()

I0000 00:00:1759094527.633067      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1759094527.633904      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "cnn_lstm_eegnet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 19, 512, 1)     │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast (Cast)               │ (None, 19, 512, 1)     │              0 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d (Conv2D)           │ (None, 19, 512, 50)    │          1,250 │ cast[0][0]             │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization       │ (None, 19, 512, 50)    │            200 │ conv2d[0][0]           │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation (Activation)   │ (None, 1, 512, 100)    │              0 │ batch_normalization[0… │
│                           │                        │                │ batch_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ depthwise_conv2d          │ (None, 1, 512, 100)    │          1,900 │ activation[0][0]       │
│ (DepthwiseConv2D)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_1     │ (None, 1, 512, 100)    │            400 │ depthwise_conv2d[0][0] │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ average_pooling2d         │ (None, 1, 50, 100)     │              0 │ activation[1][0]       │
│ (AveragePooling2D)        │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout (Dropout)         │ (None, 1, 50, 100)     │              0 │ average_pooling2d[0][… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ permute (Permute)         │ (None, 50, 1, 100)     │              0 │ dropout[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ time_distributed          │ (None, 50, 100)        │              0 │ permute[0][0]          │
│ (TimeDistributed)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm (LSTM)               │ (None, 50, 10)         │          4,440 │ time_distributed[0][0] │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_1 (LSTM)             │ (None, 10)             │            840 │ lstm[0][0]             │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 2)              │             22 │ lstm_1[0][0]           │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 9,052 (35.36 KB)

 Trainable params: 8,752 (34.19 KB)

 Non-trainable params: 300 (1.17 KB)

# Resultados - Leave 24 Subjects Out

In [6]:
import os

import pickle

with open("/kaggle/input/ieee-tdah-control-database/folds.pkl", "rb") as f:
    folds = pickle.load(f)

In [7]:
import numpy as np

results = {}

for i in range(10):
    result = train_L24O_cv(cnn_lstm_eegnet, X, y, sbjs, model_args, compile_args, folds)
    results[i] = result

--------------------------------------------------
Fold 1/5. Test subjects: ['v28p', 'v274', 'v1p', 'v231', 'v22p', 'v29p', 'v206', 'v238', 'v31p', 'v35p', 'v177', 'v200', 'v112', 'v113', 'v48p', 'v140', 'v131', 'v125', 'v55p', 'v143', 'v43p', 'v305', 'v134', 'v114']
--------------------------------------------------


E0000 00:00:1759094537.058292      19 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/cnn_lstm_eegnet_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1759094537.595373      72 cuda_dnn.cc:529] Loaded cuDNN version 90300



Epoch 34: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.

Epoch 44: ReduceLROnPlateau reducing learning rate to 0.0024999999441206455.
Epoch 49: early stopping
Restoring model weights from the end of the best epoch: 24.
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step
(1457, 2)

Fold 1 Metrics: {'accuracy': 0.8853809196980096, 'recall': 0.884078533991874, 'precision': 0.8866334319099278, 'kappa': 0.7698103283044965, 'auc': 0.9581728518132524}
Average accuracy per test subject:
  v28p: 0.4528
  v274: 1.0000
  v1p: 0.8696
  v231: 1.0000
  v22p: 1.0000
  v29p: 0.9892
  v206: 1.0000
  v238: 1.0000
  v31p: 1.0000
  v35p: 1.0000
  v177: 0.9844
  v200: 1.0000
  v112: 1.0000
  v113: 1.0000
  v48p: 0.0513
  v140: 0.6364
  v131: 1.0000
  v125: 0.9661
  v55p: 0.9815
  v143: 0.3898
  v43p: 0.9792
  v305: 1.0000
  v134: 1.0000
  v114: 1.0000
--------------------------------------------------
Fold 2/5. Test subjects: ['v18p', 'v39p', 'v234', 'v32p', 'v190', 'v6p', 'v254', 'v204', 'v24p', 

E0000 00:00:1759094881.271619      19 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/cnn_lstm_eegnet_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer



Epoch 13: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.

Epoch 23: ReduceLROnPlateau reducing learning rate to 0.0024999999441206455.
Epoch 28: early stopping
Restoring model weights from the end of the best epoch: 3.
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step
(1668, 2)

Fold 2 Metrics: {'accuracy': 0.7655875299760192, 'recall': 0.7719154933272316, 'precision': 0.7687952376561707, 'kappa': 0.5336464342816936, 'auc': 0.8145521294999467}
Average accuracy per test subject:
  v18p: 0.9583
  v39p: 0.9714
  v234: 1.0000
  v32p: 1.0000
  v190: 0.1695
  v6p: 0.0000
  v254: 0.6667
  v204: 0.0000
  v24p: 0.9841
  v183: 0.7324
  v246: 0.6951
  v219: 0.9524
  v298: 0.2647
  v41p: 1.0000
  v47p: 1.0000
  v308: 1.0000
  v52p: 1.0000
  v300: 1.0000
  v59p: 0.4444
  v299: 1.0000
  v302: 0.9265
  v51p: 1.0000
  v109: 0.6721
  v127: 0.6786
--------------------------------------------------
Fold 3/5. Test subjects: ['v215', 'v3p', 'v209', 'v37p', 'v213', 'v15p', 'v284', 'v181', 'v19p', 

...


Epoch 14: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.

Epoch 24: ReduceLROnPlateau reducing learning rate to 0.0024999999441206455.
Epoch 29: early stopping
Restoring model weights from the end of the best epoch: 4.
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step
(1768, 2)

Fold 3 Metrics: {'accuracy': 0.9117647058823529, 'recall': 0.9058425571946118, 'precision': 0.9095411413497934, 'kappa': 0.8151899039524627, 'auc': 0.9609762281246038}
Average accuracy per test subject:
  v215: 1.0000
  v3p: 0.8846
  v209: 0.9829
  v37p: 1.0000
  v213: 1.0000
  v15p: 0.7006
  v284: 1.0000
  v181: 1.0000
  v19p: 0.9888
  v34p: 1.0000
  v263: 0.9859
  v244: 1.0000
  v138: 0.0000
  v121: 1.0000
  v46p: 0.9189
  v54p: 1.0000
  v120: 1.0000
  v310: 1.0000
  v147: 0.7778
  v50p: 1.0000
  v56p: 1.0000
  v107: 1.0000
  v297: 0.5192
  v108: 1.0000
--------------------------------------------------
Fold 4/5. Test subjects: ['v227', 'v8p', 'v236', 'v14p', 'v196', 'v27p', 'v33p', 'v179', 'v173', 

E0000 00:00:1759105738.470097      19 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/cnn_lstm_eegnet_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer



Epoch 14: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.

Epoch 24: ReduceLROnPlateau reducing learning rate to 0.0024999999441206455.
Epoch 29: early stopping
Restoring model weights from the end of the best epoch: 4.
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step
(1687, 2)

Fold 4 Metrics: {'accuracy': 0.8956727919383521, 'recall': 0.8873656793055771, 'precision': 0.900219317743991, 'kappa': 0.7844114503462056, 'auc': 0.9552080285479883}
Average accuracy per test subject:
  v227: 1.0000
  v8p: 1.0000
  v236: 1.0000
  v14p: 0.6716
  v196: 0.9020
  v27p: 1.0000
  v33p: 0.9735
  v179: 1.0000
  v173: 1.0000
  v10p: 1.0000
  v265: 1.0000
  v20p: 0.8394
  v57p: 1.0000
  v45p: 0.8780
  v111: 1.0000
  v115: 1.0000
  v53p: 0.5753
  v118: 1.0000
  v123: 1.0000
  v44p: 0.3488
  v149: 0.0769
  v303: 1.0000
  v116: 1.0000
  v151: 1.0000
--------------------------------------------------
Fold 5/5. Test subjects: ['v279', 'v30p', 'v288', 'v286', 'v250', 'v12p', 'v38p', 'v25p', 'v21p', 

E0000 00:00:1759105940.131054      19 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/cnn_lstm_eegnet_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer



Epoch 16: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.

Epoch 26: ReduceLROnPlateau reducing learning rate to 0.0024999999441206455.
Epoch 31: early stopping
Restoring model weights from the end of the best epoch: 6.
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step
(1633, 2)

Fold 5 Metrics: {'accuracy': 0.9755052051439069, 'recall': 0.9752945767850978, 'precision': 0.9757659078957253, 'kappa': 0.9509574263610591, 'auc': 0.9983368607496135}
Average accuracy per test subject:
  v279: 1.0000
  v30p: 1.0000
  v288: 1.0000
  v286: 1.0000
  v250: 1.0000
  v12p: 1.0000
  v38p: 0.9895
  v25p: 0.8378
  v21p: 1.0000
  v40p: 0.9351
  v198: 0.9733
  v270: 1.0000
  v117: 0.9899
  v306: 1.0000
  v309: 1.0000
  v110: 1.0000
  v42p: 1.0000
  v58p: 1.0000
  v307: 1.0000
  v133: 0.9298
  v304: 1.0000
  v129: 0.9524
  v49p: 0.9844
  v60p: 0.6327

Cross-Validation Final Results
Individual Fold Accuracies:
  Fold 1: 0.6692
  Fold 2: 0.7488
  Fold 3: 0.9118
  Fold 4: 0.8957
  Fold 5: 0.9755

A

In [8]:
for i in range(10):
    result = results[i]
    accs = []
    for r in result:
        accs.append(r['accuracy'])
    
    print(i, '->', np.mean(accs))

0 -> 0.885110059654664
1 -> 0.8354240505514123
2 -> 0.8863725748073664
3 -> 0.8866132071442145
4 -> 0.8508860193435686
5 -> 0.8570151258962332
6 -> 0.8940285790917128
7 -> 0.8422753709901467
8 -> 0.8540151291466123
9 -> 0.8401853830914698
